# Broyles EPD Data Processing

Processing script for the Broyles compiled concrete EPD dataset (Compiled_Concrete_EPD_Data_Version_4c_Final_Published.xlsx).
Filters to relevant fields, adds GWP per cubic yard conversions, identifies SCM types (fly ash, slag), removes low-strength records, and applies IQR-based outlier removal consistent with the EC3 pipeline.

### Imports

In [1]:
import os
import pathlib
import pandas as pd
import numpy as np

### Define Outlier Removal Functions

In [2]:
def remove_outliers(df, col_names):
    """
    Remove extreme outliers based on IQR method across the full dataset.
    """
    q1 = df[col_names].quantile(0.25)
    q3 = df[col_names].quantile(0.75)
    iqr = q3 - q1

    df = df[~((df[col_names] < (q1 - 1.5 * iqr)) | (df[col_names] > (q3 + 1.5 * iqr))).any(axis=1)]

    return df.copy()


def remove_outliers_per_bucket(df, gwp_col, bucket_col):
    """
    Remove outliers within each compressive strength bucket using IQR method.
    More precise than global IQR since each strength class has a different GWP distribution.
    """
    grouped = df.groupby(bucket_col)[gwp_col]
    q1 = grouped.transform('quantile', 0.25)
    q3 = grouped.transform('quantile', 0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return df[(df[gwp_col] >= lower) & (df[gwp_col] <= upper)].copy()

### Load Data

In [3]:
# Determine repo root (works whether CWD is the notebook dir or repo root)
cwd = pathlib.Path(os.getcwd())
repo_root = cwd.parent if cwd.name == '03_processing_scripts' else cwd

excel_path = repo_root / '01_raw_data' / 'epd_data_Broyles' / 'Compiled_Concrete_EPD_Data_Version_5d_United_States_Canada_Only.xlsx'
output_path = repo_root / '02_processed_data' / 'broyles_epd_data_cleaned_with_Canada.csv'

keep_cols = [
    'Company',
    'Company Location - Street',
    'Company Location - City',
    'Company Location - State',
    'Company Location - Zip',
    'Plant',
    'Plant Location - Street',
    'Plant Location - City',
    'Plant Location - State',
    'Plant Location - Zip',
    'Country',
    'U.S. Region of Plant',
    'Metro/State (within 60 mi)',
    'EPD Program Operator',
    'EPD Date of Issue',
    'EPD Valid Until Date',
    'Mixture Label',
    'Mixture Description',
    'Concrete Compressive Strength (psi)',
    'Concrete Curation Time',
    'Declared Unit',
    'Product Components',
    'A1-A3 Global Warming Potential (kg CO2-eq)',
    'A1 GWP',
    'A2 GWP',
    'A3 GWP',
]

df = pd.read_excel(excel_path, usecols=keep_cols)

df = df.rename(columns={'Mixture Label': 'Mix Label', 'Mixture Description': 'Mix Description'})

# Coerce GWP columns to numeric (some cells use '-' as a placeholder for missing values)
gwp_raw_cols = [
    'A1-A3 Global Warming Potential (kg CO2-eq)',
    'A1 GWP',
    'A2 GWP',
    'A3 GWP',
]
for col in gwp_raw_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Loaded {len(df)} records")
print(f"Columns: {list(df.columns)}")

Loaded 49675 records
Columns: ['Company', 'Company Location - Street', 'Company Location - City', 'Company Location - State', 'Company Location - Zip', 'Plant', 'Plant Location - Street', 'Plant Location - City', 'Plant Location - State', 'Plant Location - Zip', 'Country', 'Metro/State (within 60 mi)', 'U.S. Region of Plant', 'EPD Program Operator', 'EPD Date of Issue', 'EPD Valid Until Date', 'Mix Label', 'Mix Description', 'Concrete Compressive Strength (psi)', 'Concrete Curation Time', 'Declared Unit', 'Product Components', 'A1-A3 Global Warming Potential (kg CO2-eq)', 'A1 GWP', 'A2 GWP', 'A3 GWP']


### Add GWP per Cubic Yard Columns

The declared unit in this dataset is cubic meters (m³). Convert to per cubic yard using the factor: 1 CY = 0.764555 m³, so multiply m³ values by 0.764555 to get per-CY values.

In [4]:
M3_TO_CY = 0.764555  # cubic meters per cubic yard

gwp_cols = [
    'A1-A3 Global Warming Potential (kg CO2-eq)',
    'A1 GWP',
    'A2 GWP',
    'A3 GWP'
]

for col in gwp_cols:
    df[f'{col}_per_CY'] = (df[col] * M3_TO_CY).round(2)

print("Added per-CY GWP columns:")
for col in gwp_cols:
    print(f"  {col}_per_CY")

Added per-CY GWP columns:
  A1-A3 Global Warming Potential (kg CO2-eq)_per_CY
  A1 GWP_per_CY
  A2 GWP_per_CY
  A3 GWP_per_CY


### Identify SCM Types (Fly Ash, Slag)

Parse the `Product Components` field to flag mixes containing fly ash or slag.

In [5]:
def field_contains(text, keyword):
    if pd.isna(text):
        return False
    return keyword.lower() in str(text).lower()

df['contains_fly_ash'] = df['Product Components'].apply(lambda x: field_contains(x, 'fly ash'))
df['contains_slag'] = df['Product Components'].apply(lambda x: field_contains(x, 'slag'))

print(f"Contains fly ash: {df['contains_fly_ash'].sum()} records ({df['contains_fly_ash'].mean():.1%})")
print(f"Contains slag:    {df['contains_slag'].sum()} records ({df['contains_slag'].mean():.1%})")

Contains fly ash: 22784 records (45.9%)
Contains slag:    13428 records (27.0%)


### Filter: Minimum Compressive Strength

In [6]:
before = len(df)
df = df[df['Concrete Compressive Strength (psi)'] >= 2000].copy()
print(f"Removed {before - len(df)} records with compressive strength < 2000 psi ({len(df)} remaining)")

Removed 1855 records with compressive strength < 2000 psi (47820 remaining)


### Remove Global Outliers (IQR on Full Dataset)

Remove records where `A1-A3 Global Warming Potential (kg CO2-eq)_per_CY` falls outside 1.5×IQR of the global distribution.

In [7]:
# Drop rows with NaN in the primary GWP column before outlier removal
gwp_col = 'A1-A3 Global Warming Potential (kg CO2-eq)_per_CY'

before = len(df)
df = df.dropna(subset=[gwp_col]).copy()
print(f"Dropped {before - len(df)} rows with missing A1-A3 GWP value ({len(df)} remaining)")

before = len(df)
df = remove_outliers(df, [gwp_col])
print(f"Global IQR: removed {before - len(df)} outlier rows ({len(df)} remaining)")

Dropped 0 rows with missing A1-A3 GWP value (47820 remaining)


Global IQR: removed 864 outlier rows (46956 remaining)


### Remove Per-Bucket Outliers (IQR Within Each Strength Class)

Round compressive strength to the nearest 500 psi to create buckets, then apply IQR filtering within each bucket.

In [8]:
# Drop rows with NaN compressive strength before bucketing
before = len(df)
df = df.dropna(subset=['Concrete Compressive Strength (psi)']).copy()
if before - len(df) > 0:
    print(f"Dropped {before - len(df)} rows with missing compressive strength")

# Round to nearest 500 psi for bucket assignment
df['Compressive_Strength_Bucket'] = (
    (df['Concrete Compressive Strength (psi)'] / 500).round() * 500
).astype(int)

bucket_col = 'Compressive_Strength_Bucket'
before = len(df)
df = remove_outliers_per_bucket(df, gwp_col, bucket_col)
print(f"Per-bucket IQR: removed {before - len(df)} rows ({len(df)} remaining)")

df = df.drop(columns=['Compressive_Strength_Bucket'])

Per-bucket IQR: removed 451 rows (46505 remaining)


### Geocode Plant Zip Codes to Lat/Lon

Look up latitude and longitude for each unique plant postal code using the `pgeocode` library (local GeoNames dataset — no API key required). Uses the `Country` field to route US records through the `us` GeoNames dataset (full 5-digit ZIP resolution) and Canadian records through the `ca` dataset. Note: GeoNames only publishes Canadian postal codes at the 3-character FSA (Forward Sortation Area) level (e.g. "A1A 1A6" → "A1A"), so Canadian coordinates are coarser than US ones. Deduplicates postal codes before lookup for efficiency, then merges coordinates back to the full dataframe.

In [9]:
import re
import pgeocode

nomi_us = pgeocode.Nominatim('us')
nomi_ca = pgeocode.Nominatim('ca')

def is_canada(country):
    if pd.isna(country):
        return False
    return str(country).strip().lower() in ('canada', 'can', 'ca')

def normalize_zip(z, country):
    if pd.isna(z):
        return None
    z = str(z).strip()
    if is_canada(country):
        # GeoNames only resolves Canadian postal codes at the 3-character
        # FSA (Forward Sortation Area) level, e.g. "A1A 1A6" -> "A1A"
        fsa = z.upper().replace(' ', '')[:3]
        return fsa if re.match(r'^[A-Z]\d[A-Z]$', fsa) else None
    else:
        z = z.split('-')[0].split('.')[0].strip().zfill(5)
        return z if len(z) == 5 else None

df['_is_canada'] = df['Country'].apply(is_canada)
df['_zip_norm'] = df.apply(lambda r: normalize_zip(r['Plant Location - Zip'], r['Country']), axis=1)

print(f"US records: {(~df['_is_canada']).sum():,} | Canada records: {df['_is_canada'].sum():,}")

us_zips = df.loc[~df['_is_canada'], '_zip_norm'].dropna().unique()
ca_zips = df.loc[df['_is_canada'], '_zip_norm'].dropna().unique()

us_lookup = nomi_us.query_postal_code(us_zips.tolist()).set_index('postal_code')[['latitude', 'longitude']]
ca_lookup = nomi_ca.query_postal_code(ca_zips.tolist()).set_index('postal_code')[['latitude', 'longitude']]

# US ZIPs (5 digits) and Canadian FSAs (letter-digit-letter) never collide as strings,
# so a single lookup table keyed on the normalized postal code is safe to join on.
zip_lookup = pd.concat([us_lookup, ca_lookup])
zip_lookup = zip_lookup[~zip_lookup.index.duplicated(keep='first')]

df = df.join(zip_lookup.rename(columns={'latitude': 'plant_lat', 'longitude': 'plant_lon'}), on='_zip_norm')
df = df.drop(columns=['_zip_norm', '_is_canada'])

matched = df['plant_lat'].notna().sum()
print(f"Geocoded {matched:,} of {len(df):,} records ({matched/len(df):.1%})")

US records: 44,551 | Canada records: 1,954
Geocoded 46,505 of 46,505 records (100.0%)


### Add Metro-Area Lat/Lon

Look up the metro-area centroid for each record using the `Metro/State (within 60 mi)` field and the lookup table in `02_processed_data/metro_area_lookup.csv`. For records where the field contains only a state name, fall back to the plant's zip-code lat/lon.

In [10]:
metro_lookup_path = repo_root / '02_processed_data' / 'metro_area_lookup.csv'
metro_lookup = pd.read_csv(metro_lookup_path)[['metro_area', 'is_state', 'metro_lat', 'metro_lon']]

df = df.merge(
    metro_lookup.rename(columns={'metro_area': 'Metro/State (within 60 mi)'}),
    on='Metro/State (within 60 mi)',
    how='left'
)

# For state-only entries or unmatched rows, fall back to plant lat/lon
use_plant = df['is_state'].fillna(True) | df['metro_lat'].isna()
df['metro_lat'] = df['metro_lat'].where(~use_plant, df['plant_lat'])
df['metro_lon'] = df['metro_lon'].where(~use_plant, df['plant_lon'])
df = df.drop(columns=['is_state'])

metro_area_count = (~use_plant).sum()
print(f"Records with metro-area coords: {metro_area_count:,} ({metro_area_count/len(df):.1%})")
print(f"Records using plant coords (state-only or missing): {use_plant.sum():,}")

Records with metro-area coords: 37,239 (80.1%)
Records using plant coords (state-only or missing): 9,266


### Summary

In [11]:
print(f"Final record count: {len(df)}")
print(f"Final column count: {len(df.columns)}")
print("\nColumn list:")
for col in df.columns:
    print(f"  {col}")

df.head(3)

Final record count: 46505
Final column count: 36

Column list:
  Company
  Company Location - Street
  Company Location - City
  Company Location - State
  Company Location - Zip
  Plant
  Plant Location - Street
  Plant Location - City
  Plant Location - State
  Plant Location - Zip
  Country
  Metro/State (within 60 mi)
  U.S. Region of Plant
  EPD Program Operator
  EPD Date of Issue
  EPD Valid Until Date
  Mix Label
  Mix Description
  Concrete Compressive Strength (psi)
  Concrete Curation Time
  Declared Unit
  Product Components
  A1-A3 Global Warming Potential (kg CO2-eq)
  A1 GWP
  A2 GWP
  A3 GWP
  A1-A3 Global Warming Potential (kg CO2-eq)_per_CY
  A1 GWP_per_CY
  A2 GWP_per_CY
  A3 GWP_per_CY
  contains_fly_ash
  contains_slag
  plant_lat
  plant_lon
  metro_lat
  metro_lon


,Company,Company Location - Street,Company Location - City,Company Location - State,Company Location - Zip,Plant,Plant Location - Street,Plant Location - City,Plant Location - State,Plant Location - Zip,...,A1-A3 Global Warming Potential (kg CO2-eq)_per_CY,A1 GWP_per_CY,A2 GWP_per_CY,A3 GWP_per_CY,contains_fly_ash,contains_slag,plant_lat,plant_lon,metro_lat,metro_lon
0,"404 Concrete, LLC",2547 Lithonia W Dr.,Lithonia,GA,30058,RexCon Mobile 12 Plant,2547 Lithonia W Dr.,Lithonia,GA,30058,...,152.15,135.33,3.39,13.3,True,False,33.7356,-84.1009,33.749,-84.388
1,"404 Concrete, LLC",2547 Lithonia W Dr.,Lithonia,GA,30058,RexCon Mobile 12 Plant,2547 Lithonia W Dr.,Lithonia,GA,30058,...,191.90,175.08,3.73,13.3,True,False,33.7356,-84.1009,33.749,-84.388
2,"404 Concrete, LLC",2547 Lithonia W Dr.,Lithonia,GA,30058,RexCon Mobile 12 Plant,2547 Lithonia W Dr.,Lithonia,GA,30058,...,205.67,187.32,4.85,13.3,False,False,33.7356,-84.1009,33.749,-84.388


### Save Cleaned Data

In [12]:
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} records to {output_path}")

Saved 46505 records to c:\Users\jaredf\Dropbox\EDS Course\scm-mapping\02_processed_data\broyles_epd_data_cleaned_with_Canada.csv
